# 2차 제출 재현
quantile-ensemble LightGBM (alpha 0.70/0.75/0.80 × seed) + per-group post-scale(×1.05/1.00/1.08) + IDW 공간 feature.

> 참고: 제출 이후 `features2`의 temporal feature가 제거되어, 현재 코드는 `data/2차.csv`를 **근사** 재현합니다 (상관 ≈ 0.998, 약 2% 차).

## 1. 라이브러리 · 상수 정의

**무엇을** — 넘파이/판다스와 **LightGBM**(리프 중심 성장형 그래디언트 부스팅)을 불러오고 공통 상수를 정의합니다. 1차는 HistGradientBoosting을 썼지만, 2차부터는 **분위수(quantile) 회귀**를 지원하는 LightGBM으로 교체한 것이 핵심 차이입니다.

**왜 / 어떻게**
- `TARGET_COLS` : 예측 대상 3개 발전 그룹.
- `CAPACITY_KWH` : 그룹별 설비용량. 여기서도 모델은 kWh가 아니라 **용량계수(CF=발전량/용량, 0~1)** 를 학습하고, 예측 후 용량을 곱해 복원합니다.
- `HUB = 117.0` : 터빈 허브 높이(m). 관측 풍속을 허브 높이로 외삽할 때 기준.

In [1]:
import numpy as np, pandas as pd
import lightgbm as lgb

D = "../data/"
TARGET_COLS = ["kpx_group_1", "kpx_group_2", "kpx_group_3"]
CAPACITY_KWH = {"kpx_group_1": 21600, "kpx_group_2": 21600, "kpx_group_3": 21000}
HUB = 117.0

## 2. 피처 엔지니어링 — 공통 피처 + IDW 공간 피처

원시 기상격자(LDAPS·GFS)를 예보 시각당 1행 피처로 변환합니다. 1차와 달리 **그룹 위치별 IDW(역거리가중) 공간 피처**를 추가한 것이 2차의 핵심 개선점입니다.

**행 단위 파생 (1차와 동일)**
- `_ldaps_rowfeats` / `_gfs_rowfeats` : `hypot(u,v)`로 풍속 합성, max−min으로 돌풍, 멱법칙(`ws_hub = ws50·(117/50)^alpha`)으로 허브 높이 풍속 외삽, 세제곱항(발전량 ∝ v³), 공기밀도(`p/RT`) 등.
- `calendar_features` : 시각·월·연중일을 sin/cos로 인코딩(주기성 반영).

**IDW 공간 피처 (2차 신규)** — 발전 그룹마다 실제 위치가 다르므로, 격자점들을 **그룹 중심점(`GROUP_CENTROID`)에 가까울수록 크게 가중**하여 합성합니다.
- `_grid_coords` : 격자 id별 위경도 좌표 추출.
- `_idw_weights` : 위경도를 km 거리로 환산(위도차×111, 경도차×111×cosφ) → 거리제곱의 역수 `1/(d²+0.25)`를 가중치로. 가까운 격자가 지배적이 되어 그룹 위치의 국지 바람을 더 정확히 반영합니다.
- `_time_pivot` / `_idw_series` : 변수를 (시각×격자) 표로 펼친 뒤 가중치 벡터와 내적 → 시각별 IDW 합성값.
- `build_shared` : 모든 그룹이 공유하는 평균/최대/표준편차 집계 피처.
- `build_group_extra` : 그룹별 IDW 합성 풍속·풍향(sin/cos)·풍속×풍향 상호작용·GFS 100 m 풍속 등 **그룹 전용 피처**를 생성.

In [2]:
def _ldaps_rowfeats(df):
    o = pd.DataFrame(index=df.index)
    u10, v10 = df["heightAboveGround_10_10u"], df["heightAboveGround_10_10v"]
    o["ws10"] = np.hypot(u10, v10)
    u50 = (df["heightAboveGround_50_50MUmax"] + df["heightAboveGround_50_50MUmin"]) / 2
    v50 = (df["heightAboveGround_50_50MVmax"] + df["heightAboveGround_50_50MVmin"]) / 2
    o["ws50"] = np.hypot(u50, v50)
    o["ws50max"] = np.hypot(df["heightAboveGround_50_50MUmax"], df["heightAboveGround_50_50MVmax"])
    o["ws50_gust"] = (df["heightAboveGround_50_50MUmax"] - df["heightAboveGround_50_50MUmin"]).abs() \
        + (df["heightAboveGround_50_50MVmax"] - df["heightAboveGround_50_50MVmin"]).abs()
    o["blws"] = np.hypot(df["heightAboveGround_5_XBLWS"], df["heightAboveGround_5_YBLWS"])
    alpha = np.log(np.clip(o["ws50"], 0.1, None) / np.clip(o["ws10"], 0.1, None)) / np.log(50 / 10)
    alpha = alpha.clip(-0.5, 1.0)
    o["ws_hub"] = o["ws50"] * (HUB / 50.0) ** alpha
    o["ws_hub3"] = o["ws_hub"] ** 3
    o["ws50_3"] = o["ws50"] ** 3
    o["rho"] = df["surface_0_sp"] / (287.05 * (df["heightAboveGround_2_t"]))
    o["t2"] = df["heightAboveGround_2_t"]
    o["sp"] = df["surface_0_sp"]
    o["blh"] = df["etc_0_blh"]
    o["u50"], o["v50"] = u50, v50
    return o


def _gfs_rowfeats(df):
    o = pd.DataFrame(index=df.index)
    o["gws10"] = np.hypot(df["heightAboveGround_10_10u"], df["heightAboveGround_10_10v"])
    o["gws80"] = np.hypot(df["heightAboveGround_80_u"], df["heightAboveGround_80_v"])
    o["gws100"] = np.hypot(df["heightAboveGround_100_100u"], df["heightAboveGround_100_100v"])
    o["gws100_3"] = o["gws100"] ** 3
    o["gust"] = df["surface_0_gust"]
    o["gpbl"] = np.hypot(df["planetaryBoundaryLayer_0_u"], df["planetaryBoundaryLayer_0_v"])
    o["gws850"] = np.hypot(df["isobaricInhPa_850_u"], df["isobaricInhPa_850_v"])
    o["gu100"], o["gv100"] = df["heightAboveGround_100_100u"], df["heightAboveGround_100_100v"]
    return o


def calendar_features(idx):
    dt = pd.to_datetime(pd.Series(idx))
    out = pd.DataFrame(index=idx)
    h, m, doy = dt.dt.hour.values, dt.dt.month.values, dt.dt.dayofyear.values
    out["hour_sin"] = np.sin(2 * np.pi * h / 24);  out["hour_cos"] = np.cos(2 * np.pi * h / 24)
    out["month_sin"] = np.sin(2 * np.pi * m / 12); out["month_cos"] = np.cos(2 * np.pi * m / 12)
    out["doy_sin"] = np.sin(2 * np.pi * doy / 365); out["doy_cos"] = np.cos(2 * np.pi * doy / 365)
    return out


GROUP_CENTROID = {
    "kpx_group_1": (37.28713, 128.95202),
    "kpx_group_2": (37.28225, 128.96515),
    "kpx_group_3": (37.27520, 128.97144),
}


def _grid_coords(df):
    return df.drop_duplicates("grid_id").set_index("grid_id")[["latitude", "longitude"]].sort_index()


def _idw_weights(gridc, clat, clon, power=2.0):
    dy = (gridc["latitude"] - clat) * 111.0
    dx = (gridc["longitude"] - clon) * 111.0 * np.cos(np.radians(clat))
    d2 = dx**2 + dy**2
    w = 1.0 / (d2 + 0.25)
    w = w ** (power / 2.0)
    return (w / w.sum())


def _time_pivot(rowfeats, times, gridids, col):
    rf = pd.DataFrame({"t": times.values, "g": gridids.values, "v": rowfeats[col].values})
    return rf.pivot_table(index="t", columns="g", values="v")


def _idw_series(pivot, weights):
    w = weights.reindex(pivot.columns).fillna(0.0).values
    return pd.Series(pivot.values @ w, index=pivot.index)


def build_shared(ldaps, gfs):
    lt = pd.to_datetime(ldaps["forecast_kst_dtm"]); gt = pd.to_datetime(gfs["forecast_kst_dtm"])
    lrf = _ldaps_rowfeats(ldaps); grf = _gfs_rowfeats(gfs)

    def agg(rf, times, prefix):
        x = rf.copy(); x["t"] = times.values; g = x.groupby("t")
        m = g.mean(); m.columns = [f"{prefix}_{c}_mean" for c in m.columns]
        mx = g.max(); mx = mx[[c for c in mx.columns if c != "t"]]
        mx.columns = [f"{prefix}_{c}_max" for c in mx.columns]
        sd = g.std(); sd.columns = [f"{prefix}_{c}_std" for c in sd.columns]
        keep_mx = [c for c in mx.columns if "ws" in c or "gust" in c]
        keep_sd = [c for c in sd.columns if "ws" in c]
        return pd.concat([m, mx[keep_mx], sd[keep_sd]], axis=1)

    L = agg(lrf, lt, "l"); G = agg(grf, gt, "g")
    feat = L.join(G, how="left")
    feat = feat.join(calendar_features(feat.index), how="left")
    return feat


def build_group_extra(ldaps, gfs):
    lt = pd.to_datetime(ldaps["forecast_kst_dtm"]); gt = pd.to_datetime(gfs["forecast_kst_dtm"])
    lrf = _ldaps_rowfeats(ldaps); grf = _gfs_rowfeats(gfs)
    lgc = _grid_coords(ldaps); ggc = _grid_coords(gfs)
    piv = {c: _time_pivot(lrf, lt, ldaps["grid_id"], c) for c in ["ws_hub", "ws_hub3", "ws50", "u50", "v50", "ws50_gust"]}
    gpiv = {c: _time_pivot(grf, gt, gfs["grid_id"], c) for c in ["gws100", "gu100", "gv100"]}
    out = {}
    for tgt, (clat, clon) in GROUP_CENTROID.items():
        lw = _idw_weights(lgc, clat, clon)
        gw = _idw_weights(ggc, clat, clon)
        e = pd.DataFrame(index=piv["ws_hub"].index)
        e["gws_hub"] = _idw_series(piv["ws_hub"], lw)
        e["gws_hub3"] = _idw_series(piv["ws_hub3"], lw)
        e["gws50"] = _idw_series(piv["ws50"], lw)
        e["ggust"] = _idw_series(piv["ws50_gust"], lw)
        u = _idw_series(piv["u50"], lw); v = _idw_series(piv["v50"], lw)
        ang = np.arctan2(v, u)
        e["gwd_sin"] = np.sin(ang); e["gwd_cos"] = np.cos(ang)
        e["gws_x_sin"] = e["gws50"] * e["gwd_sin"]; e["gws_x_cos"] = e["gws50"] * e["gwd_cos"]
        e["ggfs100"] = _idw_series(gpiv["gws100"], gw)
        out[tgt] = e
    return out

## 3. 데이터 로드 · 피처 생성

**무엇을 / 왜** — 학습·테스트용 원시 기상데이터·라벨·제출양식을 읽고, 공통 피처(`shared`)와 그룹별 IDW 피처(`extra`)를 학습·테스트 각각에 대해 생성합니다.
- 모든 피처 테이블의 인덱스를 예보 시각(datetime)으로 통일해, 뒤에서 라벨/제출 시각과 `reindex`·`join`으로 정확히 정렬할 수 있게 합니다.
- 그룹 전용 피처는 그룹마다 별도 테이블(`extra[tgt]`)로 보관해, 학습 시 해당 그룹 피처만 붙입니다.

In [3]:
ldaps = pd.read_csv(D+"train/ldaps_train.csv"); gfs = pd.read_csv(D+"train/gfs_train.csv")
lab = pd.read_csv(D+"train/train_labels.csv"); lab["kst_dtm"] = pd.to_datetime(lab["kst_dtm"])
shared = build_shared(ldaps, gfs); shared.index = pd.to_datetime(shared.index)
extra = build_group_extra(ldaps, gfs)
for k in extra: extra[k].index = pd.to_datetime(extra[k].index)
lab = lab.set_index("kst_dtm")

ldaps_te = pd.read_csv(D+"test/ldaps_test.csv"); gfs_te = pd.read_csv(D+"test/gfs_test.csv")
sub = pd.read_csv(D+"sample_submission.csv"); sub["forecast_kst_dtm"] = pd.to_datetime(sub["forecast_kst_dtm"])
shared_te = build_shared(ldaps_te, gfs_te); shared_te.index = pd.to_datetime(shared_te.index)
extra_te = build_group_extra(ldaps_te, gfs_te)
for k in extra_te: extra_te[k].index = pd.to_datetime(extra_te[k].index)

## 4. 분위수 앙상블 학습 · 예측 · 제출

**무엇을** — 그룹별로 LightGBM 분위수 회귀 모델을 여러 개 학습해 평균내고, 후보정(post-scale)을 적용한 뒤 제출 파일을 만듭니다.

**왜 / 어떻게**
- `kw(a, seed)` : `objective="quantile", alpha=a` → **분위수 회귀**. 평균(MSE)이 아니라 특정 분위수를 예측합니다. 여기서 `alpha`를 0.5보다 크게(0.70~0.80) 두는 것은 **중앙값보다 약간 높게 예측**하려는 의도(과소예측 페널티 대응/실측 분포 보정).
- `predict_group` : `ALPHAS`(0.70/0.75/0.80) × `seeds`(42/7/2024)의 **모든 조합을 학습·평균**하는 분위수-시드 앙상블 → 단일 분위수·단일 시드의 편향과 분산을 함께 완화.
- `SCALE` : 그룹별 후보정 배수(×1.05/1.00/1.08). 검증에서 그룹별로 체계적 과소예측이 관찰돼 이를 곱으로 보정합니다.
- 예측 CF에 스케일을 곱하고 용량을 곱해 kWh로 복원한 뒤 `[0, 용량]`으로 클리핑, `utf-8-sig`로 저장.

In [4]:
ALPHAS = [0.70, 0.75, 0.80]
SCALE = {"kpx_group_1": 1.05, "kpx_group_2": 1.00, "kpx_group_3": 1.08}

def kw(a, seed):
    return dict(objective="quantile", alpha=a, n_estimators=1200, learning_rate=0.03,
        num_leaves=63, min_child_samples=40, subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
        reg_lambda=1.0, random_state=seed, n_jobs=-1, verbose=-1)

def Xg(tgt, idx, S, E):
    return S.reindex(idx).join(E[tgt].reindex(idx), how="left")

def predict_group(tgt, Xtr, ytr, Xva, seeds):
    ps = []
    for a in ALPHAS:
        for s in seeds:
            ps.append(np.clip(lgb.LGBMRegressor(**kw(a, s)).fit(Xtr, ytr).predict(Xva), 0, 1))
    cf = np.mean(ps, axis=0) * SCALE[tgt]
    return np.clip(cf * CAPACITY_KWH[tgt], 0, CAPACITY_KWH[tgt])

tidx = sub["forecast_kst_dtm"]
out = sub[["forecast_id", "forecast_kst_dtm"]].copy()
for t in TARGET_COLS:
    m = lab[t].notna(); idx = lab.index[m]
    Xtr = Xg(t, idx, shared, extra); ytr = (lab.loc[idx, t] / CAPACITY_KWH[t]).values
    Xte = Xg(t, tidx.values, shared_te, extra_te)
    out[t] = predict_group(t, Xtr, ytr, Xte, [42, 7, 2024])

out["forecast_kst_dtm"] = pd.to_datetime(out["forecast_kst_dtm"]).dt.strftime("%Y-%m-%d %H:%M:%S")
out.to_csv(D+"submission_2차.csv", index=False, encoding="utf-8-sig")
out.head()

,forecast_id,forecast_kst_dtm,kpx_group_1,kpx_group_2,kpx_group_3
0,forecast_0001,2025-01-01 01:00:00,21168.835352,19628.213638,19220.915279
1,forecast_0002,2025-01-01 02:00:00,20544.792801,19585.696287,18907.405122
2,forecast_0003,2025-01-01 03:00:00,19832.683367,19545.763754,19361.066472
3,forecast_0004,2025-01-01 04:00:00,19368.334514,19302.466003,18481.543852
4,forecast_0005,2025-01-01 05:00:00,19745.851756,19450.568805,18946.937231


## 5. 근사 재현 검증

**무엇을 / 왜** — 생성한 예측을 기존 제출 `data/2차.csv`와 비교합니다. 1차와 달리 완전 일치가 아니라 **상관계수와 최대오차**로 확인합니다. 제출 이후 일부 temporal feature가 제거돼 결과가 미세하게 달라졌기 때문에(상관 ≈ 0.998, 약 2% 차) 근사 재현임을 정량적으로 보여줍니다.

In [5]:
ref = pd.read_csv(D+"2차.csv")
for t in TARGET_COLS:
    corr = np.corrcoef(out[t], ref[t])[0, 1]
    md = np.max(np.abs(out[t].values - ref[t].values))
    print(f"{t}: corr={corr:.4f}  maxdiff={md:.1f}")

kpx_group_1: corr=0.9976  maxdiff=3228.0
kpx_group_2: corr=0.9975  maxdiff=3641.0
kpx_group_3: corr=0.9977  maxdiff=2896.7
